# Heart Disease - Inference Demo

**MLOps Assignment 01 - Inference / model consumption**

Loads the packaged pipeline (`models/heart_disease_model.joblib`) and shows
single + batch prediction with confidence scores. This mirrors exactly what the
FastAPI `/predict` endpoint does internally.

In [1]:
import sys, json
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import joblib
import pandas as pd
from src.config import MODEL_PATH, METADATA_PATH, FEATURE_COLUMNS

model = joblib.load(MODEL_PATH)
meta = json.loads(METADATA_PATH.read_text())
print("Loaded model:", meta["best_model"])
print("Test ROC-AUC:", round(meta["metrics"]["roc_auc"], 4))

Loaded model: logistic_regression
Test ROC-AUC: 0.9665


## 1. Single-patient prediction

In [2]:
sample = {"age": 63, "sex": 1, "cp": 3, "trestbps": 145, "chol": 233,
          "fbs": 1, "restecg": 0, "thalach": 150, "exang": 0, "oldpeak": 2.3,
          "slope": 3, "ca": 0, "thal": 6}
X = pd.DataFrame([sample])[FEATURE_COLUMNS]
proba = float(model.predict_proba(X)[0, 1])
pred = int(proba >= 0.5)
print("Prediction:", "Heart disease present" if pred else "No heart disease")
print(f"P(disease) = {proba:.4f}   confidence = {(proba if pred else 1-proba):.4f}")

Prediction: No heart disease
P(disease) = 0.1254   confidence = 0.8746


## 2. Batch prediction

In [3]:
batch = pd.DataFrame([
    {"age": 41, "sex": 0, "cp": 2, "trestbps": 130, "chol": 204, "fbs": 0,
     "restecg": 2, "thalach": 172, "exang": 0, "oldpeak": 1.4, "slope": 1, "ca": 0, "thal": 3},
    {"age": 67, "sex": 1, "cp": 4, "trestbps": 160, "chol": 286, "fbs": 0,
     "restecg": 2, "thalach": 108, "exang": 1, "oldpeak": 1.5, "slope": 2, "ca": 3, "thal": 3},
    {"age": 57, "sex": 1, "cp": 4, "trestbps": 130, "chol": 131, "fbs": 0,
     "restecg": 0, "thalach": 115, "exang": 1, "oldpeak": 1.2, "slope": 2, "ca": 1, "thal": 7},
])[FEATURE_COLUMNS]
probs = model.predict_proba(batch)[:, 1]
out = batch.copy()
out["P(disease)"] = probs.round(4)
out["prediction"] = (probs >= 0.5).astype(int)
out["label"] = out["prediction"].map({0: "No disease", 1: "Disease"})
out[["age", "sex", "cp", "thalach", "ca", "P(disease)", "prediction", "label"]]

,age,sex,cp,thalach,ca,P(disease),prediction,label
0,41,0,2,172,0,0.0333,0,No disease
1,67,1,4,108,3,0.9856,1,Disease
2,57,1,4,115,1,0.9791,1,Disease


## 3. Equivalent API call

Once the service is running (`uvicorn api.main:app` or via Docker), the same
prediction is available over HTTP:

```bash
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"age":63,"sex":1,"cp":3,"trestbps":145,"chol":233,"fbs":1,"restecg":0,
       "thalach":150,"exang":0,"oldpeak":2.3,"slope":3,"ca":0,"thal":6}'
```

Response:
```json
{"prediction":1,"label":"Heart disease present","confidence":0.87,"probability_disease":0.87}
```